# SEC Filing Sentinel — Contextual AI Hackday

**Architecture:**
- **Custom Claw (Docker)** — Isolated container running daily web scraping via Brave Search, converts findings to PDFs, and pushes them to Contextual AI
- **Contextual AI** — Cloud datastore + RAG agent for storing, indexing, and querying scraped SEC documents
- **This notebook** — Sets up the backend (Part 1), builds the Docker image (Part 2), queries the agent (Part 3), connects Telegram (Part 4), and upgrades to the OpenClaw framework (Part 5)

---

## Part 1: Contextual AI Setup
Create the datastore and agent so Custom Claw has somewhere to push data and we have something to query.

In [ ]:
import os
import re
from dotenv import load_dotenv
from contextual import ContextualAI

load_dotenv()

def update_env(key: str, value: str, env_path: str = ".env"):
    """Update a key in .env — replaces if exists, appends if not."""
    with open(env_path, "r") as f:
        content = f.read()
    pattern = rf"^{re.escape(key)}=.*$"
    if re.search(pattern, content, flags=re.MULTILINE):
        content = re.sub(pattern, f"{key}={value}", content, flags=re.MULTILINE)
    else:
        content = content.rstrip("\n") + f"\n{key}={value}\n"
    with open(env_path, "w") as f:
        f.write(content)

client = ContextualAI(api_key=os.environ["CONTEXTUAL_API_KEY"])
print("Contextual AI client initialized.")

# Load existing IDs from .env so you can skip Part 1 if already set up
datastore_id = os.environ.get("DATASTORE_ID")
agent_id = os.environ.get("AGENT_ID")
if datastore_id:
    print(f"Loaded DATASTORE_ID from .env: {datastore_id}")
if agent_id:
    print(f"Loaded AGENT_ID from .env: {agent_id}")
if datastore_id and agent_id:
    print("\n✅ Existing IDs found — you can skip to Part 3 and start querying right away.")

### Create Datastore
This is where Custom Claw will push scraped SEC documents.

In [ ]:
datastore = client.datastores.create(name="SEC Filing Monitor")
datastore_id = datastore.id
print(f"Datastore created: {datastore_id}")

update_env("DATASTORE_ID", datastore_id)
print("DATASTORE_ID saved to .env")

### Create Agent (Vanilla)
A basic Contextual AI agent backed by our datastore — no system prompt, just the default behavior.

In [ ]:
agent = client.agents.create(
    name="SEC Filing Sentinel (Vanilla)",
    description="Monitors and analyzes SEC filings, regulatory changes, and compliance risks.",
    datastore_ids=[datastore_id],
)
agent_id = agent.id
print(f"Agent created: {agent_id}")

update_env("AGENT_ID", agent_id)
print("AGENT_ID saved to .env")

### Create Agent (Prompted)
Same agent, but with a system prompt that shapes how it analyzes and presents SEC filings. Run this cell **instead of** the vanilla one above to use the prompted version.

In [ ]:
agent = client.agents.create(
    name="SEC Filing Sentinel (Prompted)",
    description="Monitors and analyzes SEC filings, regulatory changes, and compliance risks.",
    datastore_ids=[datastore_id],
    system_prompt="""You are the SEC Filing Sentinel, an expert financial analyst specializing in SEC regulatory filings.

Your role:
- Analyze SEC filings (8-K, 10-K, 10-Q, 20-F, DEF 14A) with precision and clarity
- Identify material events: leadership changes, mergers, earnings surprises, restatements, auditor changes
- Flag compliance risks and unusual patterns across filings
- Always cite specific filings, dates, and companies — never speculate without evidence

When analysis requires computation, data transformation, or visualization, use code execution to:
- Calculate financial ratios, trends, or comparisons across filings
- Generate tables, charts, or summaries from structured filing data
- Parse and aggregate data points across multiple documents
- Perform date calculations, filtering, or sorting operations

Response style:
- Lead with the most important findings
- Use tables when comparing across multiple companies or filings
- When asked for summaries, prioritize material events over routine filings
- If the data doesn't contain enough information to answer, say so clearly rather than guessing
- Keep responses concise but thorough — a compliance officer should be able to act on your analysis
""",
)
agent_id = agent.id
print(f"Agent created: {agent_id}")

update_env("AGENT_ID", agent_id)
print("AGENT_ID saved to .env")

---
## Part 2: Custom Claw in Docker
Custom Claw runs in an isolated container. It uses Brave Search to find SEC filings, converts them to PDFs, and uploads them to the Contextual datastore.

### Detect Docker Environment
Run this cell first — it auto-detects whether you need `sudo` and which Compose version is available.

In [ ]:
import subprocess

def _detect_docker():
    """Auto-detect sudo requirement and Compose version."""
    # Check if docker works without sudo
    needs_sudo = subprocess.run(
        ["docker", "info"], capture_output=True, timeout=10
    ).returncode != 0

    sudo = "sudo " if needs_sudo else ""

    # Check for Compose v2 plugin first, then v1
    v2 = subprocess.run(
        f"{sudo}docker compose version".split(), capture_output=True, timeout=10
    ).returncode == 0

    compose = f"{sudo}docker compose" if v2 else f"{sudo}docker-compose"
    return sudo, compose

SUDO, COMPOSE = _detect_docker()
print(f"Docker sudo:    {'yes' if SUDO else 'no (Docker Desktop / user in docker group)'}")
print(f"Compose command: {COMPOSE}")

# Start Docker daemon if needed (workshop IDE)
if SUDO:
    subprocess.run("sudo service docker start".split(), capture_output=True)
    print("Docker daemon started (sudo service docker start)")

### Step 1: Build the Docker Image

In [ ]:
!{COMPOSE} build --no-cache openclaw

### Step 2: Launch Custom Claw
Start the container with API keys passed via .env. The container scrapes on a daily schedule and pushes documents to Contextual.

In [ ]:
!{COMPOSE} up -d --build

# Verify it's running
!{COMPOSE} ps

### Step 3: Manual Scrape (Test Run)
Trigger a one-off scrape to verify the pipeline works end-to-end before relying on the daily schedule.

In [ ]:
!{COMPOSE} exec openclaw python scrape.py --once

# Check what got uploaded
docs = client.datastores.documents.list(datastore_id=datastore_id)
print(f"\nDocuments in datastore: {len(docs.documents) if hasattr(docs, 'documents') else docs}")
for doc in (docs.documents if hasattr(docs, 'documents') else []):
    print(f"  - {doc.name} ({doc.id})")

### Step 4: Check Logs
View the container logs to confirm scraping and uploads are working.

In [ ]:
!{COMPOSE} logs --tail 50 openclaw

---
## Part 3: Query the Agent
Once Custom Claw has scraped and pushed documents, query the Contextual AI agent to analyze them.

### Check Document Parsing Status
Run this cell to see how many documents are parsed and ready vs. still processing.

In [ ]:
load_dotenv(override=True)
datastore_id = os.environ.get("DATASTORE_ID", datastore_id)

statuses = {"pending": 0, "processing": 0, "retrying": 0, "completed": 0, "failed": 0, "cancelled": 0}

cursor = None
while True:
    kwargs = {"datastore_id": datastore_id, "limit": 100}
    if cursor:
        kwargs["cursor"] = cursor
    page = client.datastores.documents.list(**kwargs)
    for doc in page.documents:
        statuses[doc.status] += 1
    cursor = page.next_cursor if hasattr(page, "next_cursor") else None
    if not cursor:
        break

total = sum(statuses.values())
print(f"Datastore: {datastore_id}")
print(f"Total documents: {total}")
print(f"  ✅ Completed:  {statuses['completed']}")
print(f"  ⏳ Processing: {statuses['processing']}")
print(f"  🕐 Pending:    {statuses['pending']}")
print(f"  🔄 Retrying:   {statuses['retrying']}")
print(f"  ❌ Failed:     {statuses['failed']}")
print(f"  🚫 Cancelled:  {statuses['cancelled']}")

if statuses["completed"] == total and total > 0:
    print(f"\n🎉 All {total} documents are parsed and ready to query!")
elif total == 0:
    print("\n⚠️  No documents found. Run Part 2 to scrape and upload SEC filings.")
else:
    not_ready = total - statuses["completed"]
    print(f"\n⏳ {not_ready} document(s) still processing. Re-run this cell to check again.")

### ⚡ Query Right Away (Pre-Populated Datastore)
Run this cell to skip scraping and connect to an existing datastore with 30+ parsed SEC filings. This lets you jump straight to querying without waiting for document ingestion.

In [ ]:
# Pre-populated datastore with 180+ parsed SEC filings — no need to wait for ingestion
PRE_POPULATED_DATASTORE_ID = "52ba14b6-47d5-42a1-a364-ea0df2e5f507"

datastore_id = PRE_POPULATED_DATASTORE_ID
update_env("DATASTORE_ID", datastore_id)

# Verify the datastore is accessible and count all documents
completed = 0
total = 0
cursor = None
while True:
    kwargs = {"datastore_id": datastore_id, "limit": 100}
    if cursor:
        kwargs["cursor"] = cursor
    page = client.datastores.documents.list(**kwargs)
    for doc in page.documents:
        total += 1
        if doc.status == "completed":
            completed += 1
    cursor = page.next_cursor if hasattr(page, "next_cursor") else None
    if not cursor:
        break

print(f"Connected to pre-populated datastore: {datastore_id}")
print(f"Documents ready: {completed}/{total}")

if completed > 0:
    print("\n✅ Ready to query! Run the agent creation cell above, then proceed to the query cells below.")

### SEC Filing Summary
Query the agent for a high-level overview of recent filings, regulatory changes, and compliance risks across all ingested documents.

In [ ]:
response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{
        "role": "user",
        "content": "Summarize any recent SEC filings, regulatory changes, or compliance risks found in the ingested documents."
    }],
)

print("=== Agent Response ===")
print(response.message.content)

print("\n=== Sources ===")
if response.attributions:
    for attr in response.attributions:
        print(f"  - {attr}")
elif response.retrieval_contents:
    for rc in response.retrieval_contents:
        print(f"  - {rc}")

### Interactive Query
Ask anything about the scraped SEC filings.

In [ ]:
# ✏️ Edit your question here:
question = "What companies filed 8-K forms this week and what were they about?"

response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{"role": "user", "content": question}],
)

print(f"\n=== Question: {question} ===\n")
print(response.message.content)

---
### Grounded RAG vs. General LLM — Side-by-Side Comparison

We test this in two rounds:

**Round 1: Fair Fight** — Questions about public SEC information that both ChatGPT (with internet access) and our SEC agent can attempt. This shows the difference in quality, sourcing, and reliability when both have a shot at the answer.

**Round 2: Unfair Fight** — Questions that require cross-referencing our specific corpus of 180+ indexed filings. ChatGPT can't do this — not because it's bad, but because no general LLM can aggregate across YOUR private document set. This is where grounded RAG becomes essential.

**The takeaway:** Round 1 shows why grounded RAG is *better*. Round 2 shows why it's *necessary*.

In [ ]:
# Setup: initialize OpenAI client for comparison queries
from openai import OpenAI

load_dotenv(override=True)
agent_id = os.environ.get("AGENT_ID", agent_id)
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_chatgpt(question: str) -> str:
    """Ask ChatGPT directly — no tools, no documents, just general knowledge."""
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        max_tokens=2048,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

def ask_sec_agent(question: str) -> str:
    """Ask the Contextual AI SEC agent — grounded in real filings."""
    response = client.agents.query.create(
        agent_id=agent_id,
        messages=[{"role": "user", "content": question}],
    )
    return response.message.content

print("Comparison functions ready. (ChatGPT vs. Contextual SEC Agent)")

#### Round 1: Fair Fight — Both Can Try
These are questions about publicly available SEC information. ChatGPT has internet access and can search. The SEC agent has the actual filings indexed. Watch for: sourcing, specificity, and confidence.

In [ ]:
Q1 = "Did Goodyear, Super Micro Computer, or Faraday Future file anything with the SEC this week? What did they file and when?"

print("=" * 60)
print("ROUND 1A: SPECIFIC COMPANY LOOKUP")
print("=" * 60)

print("\n--- ChatGPT (internet access, no document corpus) ---")
print(ask_chatgpt(Q1))

print("\n\n--- SEC Agent (180+ indexed filings) ---")
print(ask_sec_agent(Q1))

#### Round 1B: Executive changes — public info, but who has better detail?

In [ ]:
Q2 = "Which companies replaced their CFO or appointed a new board member in the past week? I need the names, dates, and what happened."

print("=" * 60)
print("ROUND 1B: EXECUTIVE CHANGES")
print("=" * 60)

print("\n--- ChatGPT (internet access, no document corpus) ---")
print(ask_chatgpt(Q2))

print("\n\n--- SEC Agent (180+ indexed filings) ---")
print(ask_sec_agent(Q2))

#### Round 2: Unfair Fight — Only Possible With Your Data
These questions require cross-referencing, counting, and aggregating across our specific corpus. No general LLM can do this — not because it's a bad model, but because it doesn't have the documents. This is the case for grounded RAG.

In [ ]:
Q3 = """Compare the auditor relationships across all recent filings in our database. 
Which companies use the same audit firm? Are there any companies that changed auditors, 
and if so, from whom to whom? Present this as a table."""

print("=" * 60)
print("ROUND 2A: CROSS-DOCUMENT AGGREGATION (auditor relationships)")
print("=" * 60)

print("\n--- ChatGPT (internet access, no document corpus) ---")
print(ask_chatgpt(Q3))

print("\n\n--- SEC Agent (180+ indexed filings) ---")
print(ask_sec_agent(Q3))

#### Round 2B: The killer question — multi-hop reasoning across the full corpus

In [ ]:
Q4 = """I'm preparing a risk briefing for our investment committee. Using the filings 
in our database:

1. Identify any companies that made leadership changes AND disclosed material risks 
   in the same filing period
2. Cross-reference: did any of those companies also have unusual items like auditor 
   changes, going-concern warnings, or restatements?
3. Flag any patterns — e.g., multiple companies in the same sector making similar 
   disclosures
4. Present your findings as an executive summary with a risk-ranked table

Cite the specific filings for every claim."""

print("=" * 60)
print("ROUND 2B: MULTI-HOP RISK BRIEFING")
print("=" * 60)

print("\n--- ChatGPT (internet access, no document corpus) ---")
print(ask_chatgpt(Q4))

print("\n\n--- SEC Agent (180+ indexed filings) ---")
print(ask_sec_agent(Q4))

---
## Part 4: Telegram Integration
Connect Custom Claw to Telegram so you can interact with it from your phone. We'll build this up step by step:

1. **4a: One-shot briefing** — Send a single SEC summary to Telegram
2. **4b: SEC chat bot** — Chat back and forth with the SEC agent via Telegram (runs in Docker, always on)
3. **4c: Full Custom Claw agent** — Upgrade the bot with ChatGPT + web search + SEC filings (the full agent experience)

> **First time?** See the [Telegram Setup](README.md#telegram-setup) section in the README for step-by-step instructions on creating a bot with @BotFather and getting your chat ID. You'll need `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` in your `.env` before continuing.

### 4a: One-Shot Briefing
The simplest integration — query the SEC agent and push the summary to your phone. Good for daily briefings.

In [ ]:
import requests as req

load_dotenv(override=True)
TELEGRAM_BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID")

def send_telegram(message: str):
    """Send a message via Telegram bot."""
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    resp = req.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": message, "parse_mode": "Markdown"})
    resp.raise_for_status()
    print("Telegram message sent!")

# Query the agent and send the summary to Telegram
response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{"role": "user", "content": "Give a brief summary of the most important SEC filings from today's scrape. Keep it under 500 words."}],
)

send_telegram(f"*SEC Filing Briefing*\n\n{response.message.content}")

### 4b: SEC Chat Bot
Now let's make it interactive — chat back and forth with the SEC agent via Telegram. This runs inside Docker so it's always on, even when the notebook is closed. It only talks to your Contextual SEC agent (no web search, no ChatGPT).

Run this cell to launch the SEC chat bot, then message your Telegram bot with any SEC question.

In [ ]:
# Launch the SEC-only chat bot in Docker
# (stops the full agent if it was running — only one bot can poll Telegram at a time)
!{COMPOSE} up -d --build openclaw
!{COMPOSE} run -d --name sec-chat-bot openclaw python telegram_sec_bot.py

print("\nSEC chat bot is live! Message your Telegram bot.")
print(f"To stop: {COMPOSE} stop sec-chat-bot")

### 4c: Full Custom Claw Agent
The final evolution — ChatGPT acts as the brain and decides which tools to use. The bot can now:
- Query your SEC filing database
- Search the live web via Brave
- Answer general questions from ChatGPT's own knowledge
- Combine multiple tools in a single answer

Once running, close the notebook — the bot stays alive as long as Docker is running.

**Try these in Telegram:**

| Question | What happens |
|----------|-------------|
| "What 8-K filings came in this week?" | Queries your SEC datastore |
| "What's Tesla's stock price right now?" | Searches the live web via Brave |
| "Compare NVDA and AMD market caps" | Web search for current data |
| "Any new proxy statements in our database?" | Queries SEC datastore |
| "Summarize the latest news about the SEC and crypto regulation" | Web search for current news |
| "What is a 10-K filing?" | Answers directly from ChatGPT's knowledge |
| "Find recent executive changes in our 8-K filings and look up those companies' stock performance" | Uses both tools — SEC filings + web search |

In [ ]:
# Stop the SEC-only bot (only one bot can poll Telegram at a time)
!{SUDO}docker stop sec-chat-bot 2>/dev/null; {SUDO}docker rm sec-chat-bot 2>/dev/null

# Rebuild and launch both services (scraper + full agent)
!{COMPOSE} build --no-cache
!{COMPOSE} up -d
!{COMPOSE} ps

# Check the bot started
print("\n--- Full agent bot logs ---")
!{COMPOSE} logs --tail 10 telegram-bot

---
## Part 5: From Custom Claw to OpenClaw

You just built a working agent in ~230 lines of Python. It routes questions to the right tool, talks to your SEC database, and chats via Telegram. **That's Custom Claw** — and it proves the concept.

But notice what it *doesn't* do: no memory between conversations, no dashboard, no multi-channel support, no skill marketplace, no scheduling. Building all of that yourself is a massive effort.

**OpenClaw** is an open-source agent framework that gives you all of that out of the box — same Contextual AI backend, same data, more capabilities.

| | Custom Claw | OpenClaw |
|--|-------------|----------|
| **Code** | ~230 lines of Python | Config + skill Markdown files |
| **Agent loop** | Hand-written (ChatGPT + function calling) | Built-in agent runtime |
| **Tool system** | Manually wired Python functions | Portable Markdown-based skills |
| **Telegram** | Custom polling loop | Built-in channel management |
| **LLM** | Hardcoded GPT-4o | Configurable (GPT-4o, Claude, Gemini, etc.) |
| **Memory** | None (stateless) | Persistent (Markdown on disk) |
| **Dashboard** | None | Web UI on port 18789 |
| **Multi-channel** | Telegram only | Telegram, Discord, Slack, WhatsApp, etc. |
| **Skills** | 2 (hardcoded) | 50+ built-in + custom + community marketplace |
| **Scheduling** | None | Built-in cron |

**Custom Claw proves the concept. OpenClaw makes it production-ready.**

### Switch to OpenClaw
Run this cell to stop Custom Claw and launch the OpenClaw framework. Your Telegram bot token, Contextual AI agent, and all API keys carry over automatically.

In [ ]:
import time

print("=== Switching from Custom Claw to OpenClaw ===\n")

# Step 1: Stop Custom Claw
print("1. Stopping Custom Claw...")
!{COMPOSE} stop telegram-bot 2>/dev/null || true
!{SUDO}docker stop sec-chat-bot 2>/dev/null; {SUDO}docker rm sec-chat-bot 2>/dev/null
print("   Custom Claw stopped.\n")

# Step 2: Copy .env into openclaw-agent/
print("2. Syncing API keys to OpenClaw...")
import shutil
shutil.copy(".env", "openclaw-agent/.env")
print("   Keys synced.\n")

# Step 3: Pull and start OpenClaw gateway
OPENCLAW_COMPOSE = f"{SUDO}docker compose -f openclaw-agent/docker-compose.yml" if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0 else f"{SUDO}docker-compose -f openclaw-agent/docker-compose.yml"

print(f"   Using: {OPENCLAW_COMPOSE}")
print("3. Starting OpenClaw gateway...")
!{OPENCLAW_COMPOSE} up -d openclaw-gateway

print("\n   Waiting for gateway to be healthy...")
for i in range(15):
    result = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if result.returncode == 0:
        print("   ✅ Gateway is healthy!")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")
else:
    print("   ⚠️  Gateway didn't start in time. Check logs:")
    !{OPENCLAW_COMPOSE} logs --tail 20 openclaw-gateway

### Configure OpenClaw
Set GPT-4o as the model and connect Telegram.

In [ ]:
# Set OpenAI GPT-4o as the LLM
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set agents.defaults.model openai/gpt-4o

# Bind to all interfaces so the dashboard is accessible from the host
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set gateway.bind lan

# Connect Telegram using the same bot token from .env
telegram_token = os.environ.get("TELEGRAM_BOT_TOKEN", "")
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels add --channel telegram --token {telegram_token}

# Restart to apply all config
print("\nRestarting gateway to apply config...")
!{OPENCLAW_COMPOSE} restart openclaw-gateway

import time; time.sleep(5)

# Verify
print("\n=== OpenClaw Status ===")
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels status

### Pair Telegram
OpenClaw requires a one-time pairing for security. Send any message to your Telegram bot — it will reply with a **pairing code**. Paste that code below and run the cell.

In [ ]:
# ✏️ Paste your pairing code from Telegram here:
PAIRING_CODE = "XXXXXXXX"

!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs pairing approve telegram {PAIRING_CODE}

### Verify Skills
Confirm our custom skills (SEC filings + Brave Search) are loaded alongside OpenClaw's 50+ built-in skills.

In [ ]:
# Show all ready skills (including our custom ones)
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs skills list 2>&1 | grep "ready"

### Test OpenClaw
Send a message through the OpenClaw agent directly from the CLI. You can also just message your Telegram bot — it's already connected.

**Try these in Telegram:**

| Message | What happens |
|---------|-------------|
| "What companies filed 8-K forms this week?" | `query-sec-filings` skill → Contextual AI |
| "What's Tesla's stock price right now?" | `brave-search` skill → Brave API |
| "What is a 10-K filing?" | Answers directly from GPT-4o |
| "What's the weather in New York?" | Built-in `weather` skill |

In [ ]:
# Test: Query SEC filings via OpenClaw agent CLI
import json as _json

result = subprocess.run(
    f"{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent --agent main --json --message".split() + ["What companies filed 8-K forms this week? Keep it brief."],
    capture_output=True, text=True, timeout=120
)

try:
    data = _json.loads(result.stdout)
    print(data.get("text", result.stdout))
except _json.JSONDecodeError:
    print(result.stdout)

if result.stderr:
    print(f"\n(stderr: {result.stderr[:200]})")

In [ ]:
# Test: Live web search via OpenClaw agent CLI
result = subprocess.run(
    f"{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent --agent main --json --message".split() + ["What is Tesla's stock price right now?"],
    capture_output=True, text=True, timeout=120
)

try:
    data = _json.loads(result.stdout)
    print(data.get("text", result.stdout))
except _json.JSONDecodeError:
    print(result.stdout)

if result.stderr:
    print(f"\n(stderr: {result.stderr[:200]})")

### OpenClaw Dashboard
OpenClaw includes a web UI for monitoring sessions, viewing logs, and managing skills. The dashboard requires device pairing — run the cells below to get the URL and approve access.

In [ ]:
# Step 1: Get the dashboard URL — open this in your browser
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs dashboard --no-open
print("\nOpen the URL above in your browser (use 'localhost' instead of '127.0.0.1').")
print("Click Connect. If you see 'pairing required', run the next cell.")

In [ ]:
# Step 2: Approve all pending device pairing requests
import json as _json

result = subprocess.run(
    f"{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs devices list --json".split(),
    capture_output=True, text=True, timeout=15
)

try:
    devices = _json.loads(result.stdout)
    pending = devices.get("pending", [])
    if not pending:
        print("No pending pairing requests. Dashboard should connect.")
    else:
        print(f"Approving {len(pending)} pending request(s)...")
        for req in pending:
            req_id = req.get("requestId", "")
            if req_id:
                subprocess.run(
                    f"{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs devices approve {req_id}".split(),
                    capture_output=True, text=True, timeout=15
                )
                print(f"  Approved: {req_id}")
        print("\nDone! Go back to the dashboard and click Connect again.")
except _json.JSONDecodeError:
    # Fallback: non-JSON output, try approving manually
    print("Could not parse device list. Listing devices:")
    !{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs devices list

---
### Cleanup
Stop the OpenClaw gateway when you're done.

In [ ]:
# Stop OpenClaw
!{OPENCLAW_COMPOSE} down
print("OpenClaw gateway stopped.")